In [1]:
#!/usr/bin/env python3
# Readers-Writers problem using a MONITOR (writers get priority)
#
# Run:  python readers_writers_monitor.py --readers 4 --writers 2 --rounds 3
# Then open generated_html/readers_writers_monitor.html in a browser.
#
# Python has no built-in "monitor" keyword like Java's synchronized, so the
# monitor is built by hand as a class:
#   - one Lock that guards ALL the monitor's variables (only one thread can be
#     inside a monitor method at a time -> that is the mutual exclusion part)
#   - two condition variables on that lock:
#         can_read   -> readers sleep here while a writer is active or waiting
#         can_write  -> writers sleep here while someone is reading/writing
#
# Monitor variables:  active_readers, writing (True/False), waiting_writers
#
# Critical section = the actual read/write of the shared value. It happens
# OUTSIDE the monitor lock (otherwise readers could never read together), but
# the monitor decides who is allowed in via begin_read/end_read etc.
#
# Writers have priority here: a new reader waits if any writer is waiting.
# This stops writer starvation, but now readers could starve if writers keep coming.

import argparse
import json
import os
import random
import threading
import time
import webbrowser


class Trace:
    """Keeps a list of everything that happens, the HTML is made from this list."""

    def __init__(self, names):
        self.lock = threading.Lock()      # only so two threads don't write to the log together
        self.t0 = time.time()
        self.states = {n: "THINKING" for n in names}
        self.events = []
        self.violations = 0

    def log(self, who, state, msg, value, read_count):
        with self.lock:
            self.states[who] = state
            readers = len([a for a in self.states if a[0] == "R" and self.states[a] == "READING"])
            writers = len([a for a in self.states if a[0] == "W" and self.states[a] == "WRITING"])

            # a writer should never be in together with anyone else
            if writers > 1 or (writers == 1 and readers > 0):
                self.violations += 1

            self.events.append({
                "t": round(time.time() - self.t0, 3),
                "actor": who,
                "state": state,
                "message": msg,
                "states": dict(self.states),
                "value": value,
                "read_count": read_count,
                "active_readers": readers,
                "active_writers": writers,
            })


class Shared:
    def __init__(self):
        self.value = 0        # the data everyone is fighting over


class ReadWriteMonitor:
    """The monitor. All the waiting/signalling logic lives in here."""

    def __init__(self, shared, trace):
        self.shared = shared
        self.trace = trace
        self.lock = threading.Lock()
        self.can_read = threading.Condition(self.lock)
        self.can_write = threading.Condition(self.lock)

        self.active_readers = 0
        self.writing = False
        self.waiting_writers = 0

    def begin_read(self, name):
        with self.lock:
            self.trace.log(name, "WAITING", "wants to read", self.shared.value, self.active_readers)
            # sleep while a writer is using it or a writer is queued (writer priority)
            while self.writing or self.waiting_writers > 0:
                self.can_read.wait()
            self.active_readers += 1
            self.trace.log(name, "READING", "started reading", self.shared.value, self.active_readers)

    def end_read(self, name):
        with self.lock:
            self.active_readers -= 1
            if self.active_readers == 0:
                self.can_write.notify()       # last reader out -> wake one writer
            self.trace.log(name, "THINKING", "done reading", self.shared.value, self.active_readers)

    def begin_write(self, name):
        with self.lock:
            self.waiting_writers += 1
            self.trace.log(name, "WAITING", "wants to write", self.shared.value, self.active_readers)
            while self.writing or self.active_readers > 0:
                self.can_write.wait()
            self.waiting_writers -= 1
            self.writing = True
            self.trace.log(name, "WRITING", "started writing", self.shared.value, self.active_readers)

    def end_write(self, name):
        with self.lock:
            self.writing = False
            # another writer waiting? let it go first, otherwise wake all the readers
            if self.waiting_writers > 0:
                self.can_write.notify()
            else:
                self.can_read.notify_all()
            self.trace.log(name, "THINKING", "done writing", self.shared.value, self.active_readers)


def reader(num, monitor, shared, trace, rounds, delay):
    name = "R" + str(num)
    for _ in range(rounds):
        time.sleep(random.uniform(0.05, delay))   # doing something else

        monitor.begin_read(name)

        # critical section (many readers can be here together)
        seen = shared.value
        time.sleep(random.uniform(0.1, delay))
        trace.log(name, "READING", "read value = " + str(seen), shared.value, monitor.active_readers)

        monitor.end_read(name)


def writer(num, monitor, shared, trace, rounds, delay):
    name = "W" + str(num)
    for _ in range(rounds):
        time.sleep(random.uniform(0.1, delay * 1.5))

        monitor.begin_write(name)

        # critical section (only this writer is inside)
        old = shared.value
        time.sleep(random.uniform(0.1, delay))
        shared.value = old + 1
        trace.log(name, "WRITING", "wrote " + str(old) + " -> " + str(shared.value),
                  shared.value, monitor.active_readers)

        monitor.end_write(name)


# ---------------- HTML part ----------------
# The page below is just a player. All the data (EVENTS) is filled in from the trace.

PAGE = """<!DOCTYPE html>
<html><head><meta charset="utf-8">
<title>Readers-Writers (Monitor)</title>
<style>
 body{font-family:Arial,sans-serif;background:#f3f4f7;margin:0;padding:18px}
 .bar{background:#fff;padding:10px;border-radius:6px;margin-bottom:12px}
 button{padding:5px 12px;margin-right:4px;cursor:pointer}
 .box{background:#fff;border-radius:6px;padding:10px}
 .two{display:grid;grid-template-columns:1fr 1fr;gap:12px;margin-top:12px}
 #arena{position:relative;height:330px;background:#fff;border-radius:6px}
 .zone{position:absolute;top:10px;bottom:10px;border:2px dashed #bbb;border-radius:8px;transition:background .3s,border-color .3s}
 .zone > span{position:absolute;top:6px;left:10px;font-weight:bold;font-size:13px;color:#555}
 #zIdle{left:2%;width:20%}
 #zWait{left:25%;width:20%}
 #zRes{left:48%;width:50%}
 #zRes.reading{border:2px solid #1f9d55;background:#e8f7ee}
 #zRes.writing{border:2px solid #d33f49;background:#fdecee}
 #resInfo{position:absolute;bottom:8px;left:0;right:0;text-align:center;font-size:14px}
 #val{font-weight:bold;font-size:22px}
 .a{position:absolute;width:46px;height:46px;border-radius:50%;color:#fff;font-weight:bold;
    display:flex;align-items:center;justify-content:center;
    transition:left .5s ease,top .5s ease,background .3s;box-shadow:0 1px 4px rgba(0,0,0,.3)}
 .a.W{border-radius:8px}
 .THINKING{background:#8a94a6}.WAITING{background:#e6a100}.READING{background:#1f9d55}.WRITING{background:#d33f49}
 #log{height:230px;overflow-y:auto;font-family:monospace;font-size:12px}
 #log div{padding:2px;border-bottom:1px solid #eee}
 #log div.now{background:#fff3c4}
 #line{display:flex;flex-wrap:wrap;gap:2px;margin-top:6px}
 .tick{width:10px;height:10px;background:#ccc;cursor:pointer}
 .tick.READING{background:#1f9d55}.tick.WRITING{background:#d33f49}.tick.WAITING{background:#e6a100}.tick.THINKING{background:#8a94a6}
 .tick.now{outline:2px solid #000}
 .key span{display:inline-block;color:#fff;padding:2px 8px;border-radius:4px;font-size:12px;margin-right:4px}
</style></head><body>
<h2>Readers-Writers Problem - Monitor Based</h2>
<p>__COUNT__ events recorded, __VIOL__ mutual exclusion violations. Circles = readers, squares = writers.</p>
<div class="bar">
 <button onclick="play()">Start / Replay</button><button onclick="pause()">Pause</button>
 <button onclick="step()">Step</button><button onclick="reset()">Reset</button>
 Speed <input type="range" id="speed" min="1" max="10" value="6" oninput="changeSpeed()">
 <input type="range" id="scrub" min="0" max="0" value="0" style="width:200px" oninput="jump(this.value)">
 <span id="pos"></span>
 <span class="key"><span class="THINKING">idle</span><span class="WAITING">waiting</span><span class="READING">reading</span><span class="WRITING">writing</span></span>
</div>
<div id="arena">
 <div class="zone" id="zIdle"><span>Idle / thinking</span></div>
 <div class="zone" id="zWait"><span>Waiting (on condition var)</span></div>
 <div class="zone" id="zRes"><span>Shared resource (critical section)</span>
  <div id="resInfo"><div id="status">FREE</div>value = <span id="val">0</span> &nbsp; readers inside = <span id="rc">0</span></div>
 </div>
</div>
<div class="two">
 <div class="box"><b>Timeline</b><div id="line"></div></div>
 <div class="box"><b>Event log</b><div id="log"></div></div>
</div>
<script>
var EVENTS = __DATA__;
var names = EVENTS.length ? Object.keys(EVENTS[0].states) : [];
var idx = -1, timer = null;
function $(id){ return document.getElementById(id); }

$('scrub').max = EVENTS.length;
names.forEach(function(n){
  var d = document.createElement('div');
  d.className = 'a ' + n[0] + ' THINKING'; d.id = 'a_' + n;
  d.textContent = n;
  $('arena').appendChild(d);
});
EVENTS.forEach(function(e, i){
  var l = document.createElement('div');
  l.id = 'l' + i;
  l.textContent = '[' + e.t.toFixed(2) + 's] ' + e.actor + ': ' + e.message;
  $('log').appendChild(l);
  var t = document.createElement('div');
  t.className = 'tick ' + e.state; t.id = 't' + i; t.title = e.actor + ' ' + e.state;
  t.onclick = function(){ jump(i + 1); };
  $('line').appendChild(t);
});

// put the actors that are in one state inside one zone, in a small grid
function place(list, zone){
  var z = $(zone);
  var cols = Math.max(1, Math.floor((z.offsetWidth - 12) / 56));
  list.forEach(function(n, i){
    var a = $('a_' + n);
    a.style.left = (z.offsetLeft + 12 + (i % cols) * 56) + 'px';
    a.style.top = (z.offsetTop + 34 + Math.floor(i / cols) * 56) + 'px';
  });
}

function draw(){
  document.querySelectorAll('.now').forEach(function(x){ x.classList.remove('now'); });
  var e = idx >= 0 ? EVENTS[idx] : null;
  var idle = [], wait = [], inside = [];
  names.forEach(function(n){
    var s = e ? e.states[n] : 'THINKING';
    $('a_' + n).className = 'a ' + n[0] + ' ' + s;
    if (s == 'THINKING') idle.push(n);
    else if (s == 'WAITING') wait.push(n);
    else inside.push(n);
  });
  place(idle, 'zIdle'); place(wait, 'zWait'); place(inside, 'zRes');

  if (!e){
    $('zRes').className = 'zone'; $('status').textContent = 'FREE';
    $('val').textContent = 0; $('rc').textContent = 0;
  } else {
    $('val').textContent = e.value;
    $('rc').textContent = e.read_count;
    if (e.active_writers){ $('zRes').className = 'zone writing'; $('status').textContent = 'LOCKED BY A WRITER (exclusive)'; }
    else if (e.active_readers){ $('zRes').className = 'zone reading'; $('status').textContent = e.active_readers + ' READER(S) INSIDE (shared)'; }
    else { $('zRes').className = 'zone'; $('status').textContent = 'FREE'; }
    $('l' + idx).classList.add('now');
    $('l' + idx).scrollIntoView({block: 'nearest'});
    $('t' + idx).classList.add('now');
  }
  $('scrub').value = idx + 1;
  $('pos').textContent = (idx + 1) + ' / ' + EVENTS.length;
}
function delay(){ return 1100 - $('speed').value * 100; }
function step(){ if (idx < EVENTS.length - 1){ idx++; draw(); } else pause(); }
function play(){
  pause();
  if (idx >= EVENTS.length - 1) idx = -1;   // replay from the start
  timer = setInterval(step, delay());
}
function changeSpeed(){ if (timer){ play(); } }
function pause(){ if (timer) clearInterval(timer); timer = null; }
function reset(){ pause(); idx = -1; draw(); }
function jump(n){ idx = parseInt(n) - 1; draw(); }

draw();
window.onresize = draw;
setTimeout(play, 600);   // starts by itself
</script></body></html>
"""


def make_html(trace, path):
    html = PAGE.replace("__DATA__", json.dumps(trace.events))
    html = html.replace("__COUNT__", str(len(trace.events)))
    html = html.replace("__VIOL__", str(trace.violations))
    folder = os.path.dirname(os.path.abspath(path))
    if not os.path.exists(folder):
        os.makedirs(folder)
    with open(path, "w", encoding="utf-8") as f:
        f.write(html)


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--readers", type=int, default=4)
    p.add_argument("--writers", type=int, default=2)
    p.add_argument("--rounds", type=int, default=3)
    p.add_argument("--delay", type=float, default=0.4)
    p.add_argument("--seed", type=int, default=None)
    p.add_argument("--output", default="generated_html/readers_writers_monitor.html")
    # parse_known_args so it also works inside Jupyter (which passes extra --f=... arguments)
    args, _ = p.parse_known_args()

    if args.seed is not None:
        random.seed(args.seed)

    names = ["R" + str(i) for i in range(1, args.readers + 1)]
    names += ["W" + str(i) for i in range(1, args.writers + 1)]

    shared = Shared()
    trace = Trace(names)
    monitor = ReadWriteMonitor(shared, trace)

    threads = []
    for i in range(1, args.readers + 1):
        threads.append(threading.Thread(target=reader, args=(i, monitor, shared, trace, args.rounds, args.delay)))
    for i in range(1, args.writers + 1):
        threads.append(threading.Thread(target=writer, args=(i, monitor, shared, trace, args.rounds, args.delay)))

    print("Starting", args.readers, "readers and", args.writers, "writers")
    for t in threads:
        t.start()
    for t in threads:
        t.join()

    for e in trace.events:
        print("[%6.2fs] %-3s %-8s %s" % (e["t"], e["actor"], e["state"], e["message"]))

    expected = args.writers * args.rounds
    print("\nFinal value:", shared.value, "(expected", expected, ")")
    print("Violations:", trace.violations)

    # sanity checks
    assert shared.value == expected, "lost update!"
    assert trace.violations == 0, "mutual exclusion broken!"

    make_html(trace, args.output)
    full_path = os.path.abspath(args.output)
    print("HTML written to", full_path)

    # try to open it in the default browser (copy the path above if this doesn't work)
    try:
        webbrowser.open("file:///" + full_path.replace("\\", "/"))
    except Exception:
        pass


if __name__ == "__main__":
    main()

Starting 4 readers and 2 writers
[  0.05s] R2  WAITING  wants to read
[  0.05s] R2  READING  started reading
[  0.12s] R3  WAITING  wants to read
[  0.12s] R3  READING  started reading
[  0.14s] R1  WAITING  wants to read
[  0.14s] R1  READING  started reading
[  0.15s] W2  WAITING  wants to write
[  0.29s] R2  READING  read value = 0
[  0.29s] R2  THINKING done reading
[  0.31s] R4  WAITING  wants to read
[  0.31s] W1  WAITING  wants to write
[  0.34s] R1  READING  read value = 0
[  0.34s] R1  THINKING done reading
[  0.35s] R3  READING  read value = 0
[  0.35s] R3  THINKING done reading
[  0.35s] W2  WRITING  started writing
[  0.49s] W2  WRITING  wrote 0 -> 1
[  0.49s] W2  THINKING done writing
[  0.49s] W1  WRITING  started writing
[  0.54s] R2  WAITING  wants to read
[  0.64s] R3  WAITING  wants to read
[  0.73s] R1  WAITING  wants to read
[  0.80s] W2  WAITING  wants to write
[  0.84s] W1  WRITING  wrote 1 -> 2
[  0.84s] W1  THINKING done writing
[  0.84s] W2  WRITING  started wr